# CO2 Prediction: Random Forest vs Neural Network

Goal: Predict the target value from 3 input features and compare two models.

## Dataset 
-Source: `co2.csv`

-Features: `engine` , `cylandr` , `fuelcomb`

-Target: `out1`

## 1: Import Libraries

In [27]:
import pandas as pd
from sklearn.metrics import mean_absolute_error , r2_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
from tensorflow.keras.layers import Dense , Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping , ReduceLROnPlateau

## 2: Load the Dataset
Load `co2.csv` and look at the first rows with `head()`

In [28]:
df = pd.read_csv("co2.csv")
df.head()

,engine,cylandr,fuelcomb,out1
0,2.0,4,8.5,196
1,2.4,4,9.6,221
2,1.5,4,5.9,136
3,3.5,6,11.1,255
4,3.5,6,10.6,244


## 3: Check Missing Values
Check for NaN values , becuase missing data can cause errors in training

In [29]:
df.isna().sum()

engine      0
cylandr     0
fuelcomb    0
out1        0
dtype: int64

## 4: Define Features and Target 
`x` contains the 3 input features and `y` is the value we want to predict

In [30]:
x = df.drop("out1",axis=1)
y = df["out1"]

## 5: Train/Test Split
Split the data into training and test sets, so the model is evaluted on data it has not seen.

In [31]:
x_train , x_test , y_train , y_test = train_test_split(x,y,test_size=0.2,random_state=42)

## 6: Random Forest Model 
Train a `RandomForestRegressor` and predict on the test set.

In [32]:
model_rf = RandomForestRegressor(random_state=42)
model_rf.fit(x_train,y_train)
y_pred = model_rf.predict(x_test)

## 7: Evaluted Random Forest
R² shows how much of the variation the model explains.

MAE is the average error in the orginal units.

In [33]:
print(r2_score(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))

0.9523421799831732
7.344583095238098


## 8: Cross-Validation 
Use 5-fold cross-validation to check that the result is not just luck from one split.

In [34]:
scores = cross_val_score(model_rf,x,y,cv=5,scoring='r2')
print(scores)
print(scores.mean())

[0.86354803 0.95550242 0.98322368 0.86329238 0.94922769]
0.9229588388404932


## 9: Neural Network Model
Build a `Sequential` network wiyh 3 hidden layers.
EarlyStopping and ReduseLROnPlateau are used to stop training when validation loss stops improving.

In [42]:
model_seq = Sequential()
model_seq.add(Dense(200,activation='relu',input_dim = 3))
model_seq.add(Dense(150,activation='relu'))
model_seq.add(Dense(100,activation='relu'))
model_seq.add(Dense(1,activation="linear"))

model_seq.compile(
    optimizer = 'adam',
    loss = 'mse',
    metrics = ['r2_score']
)
callbacks = [
    EarlyStopping(monitor='val_loss',patience=20,restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=8)
]



model_seq.fit(x_train,y_train,epochs = 500,batch_size = 32, validation_split = 0.2, callbacks=callbacks)


Epoch 1/500


c:\Users\snid\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 72176.0078 - r2_score: -16.3584 - val_loss: 73895.4375 - val_r2_score: -18.0790 - learning_rate: 0.0010
Epoch 2/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 63721.0703 - r2_score: -14.3250 - val_loss: 60994.6367 - val_r2_score: -14.7482 - learning_rate: 0.0010
Epoch 3/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 46726.0664 - r2_score: -10.2377 - val_loss: 36402.2930 - val_r2_score: -8.3987 - learning_rate: 0.0010
Epoch 4/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 20590.9961 - r2_score: -3.9521 - val_loss: 6949.9932 - val_r2_score: -0.7944 - learning_rate: 0.0010
Epoch 5/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2667.4934 - r2_score: 0.3585 - val_loss: 3190.2644 - val_r2_score: 0.1763 - learning_rate: 0.0010
Epoch 6/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2934.8181 - r2_score: 0.2942 - val_loss: 1291.7753 - val_r2_score: 0.6665 - learning_rate: 0.0010
Epoch 7/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/ste

## 10: Evalute Neural Network
Evaluate the network on the same test set, so the comparison with Random 
forest is fair.

In [44]:
y_pred = model_seq.predict(x_test)
print(r2_score(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
0.9010911583900452
13.56213092803955


## 11: Conclusion

|Model|R²|MAE|
|-----|--|---|
|Random Forest|95.23|7.34|
|Neural Network|90.10|13.56|

## Possible Improvements
-Scale the data with `StandardScaler` before training the neural network.

-Try a smaller network with Dropout.

-Tune the Random Forest hyperparameters.